# MVP2 Pingpong — PYNQ Test Notebook

Tests the full pipeline: CORDIC source → FDTD solver (64×64) → field magnitude → ping-pong BRAMs → PS readback.

**Run cells top to bottom. Each cell prints a PASS/FAIL result.**

---
## Cell 1 — Load bitstream

In [ ]:
from pynq import Bitstream
import time

bs = Bitstream('/home/xilinx/mvp2_pingpong.bit')
bs.download()
print('✓ Bitstream loaded')

---
## Cell 2 — AXI GPIO helpers

In [ ]:
from pynq import MMIO
import numpy as np

class GPIO:
    """Dual-channel AXI GPIO wrapper."""
    def __init__(self, base):
        self.m = MMIO(base, 0x10000)
    def ch1_w(self, v):  self.m.write(0x00, int(v) & 0xFFFFFFFF)
    def ch2_w(self, v):  self.m.write(0x08, int(v) & 0xFFFFFFFF)
    def ch1_r(self):     return self.m.read(0x00)
    def ch2_r(self):     return self.m.read(0x08)

ctrl   = GPIO(0x41200000)  # control  PS->PL
stat   = GPIO(0x41210000)  # status   PL->PS
smag_a = GPIO(0x41220000)  # smag_a BRAM
smag_b = GPIO(0x41230000)  # smag_b BRAM

PHASE_STEP  = 200
AMPLITUDE   = 4096   # 0.5 in Q3.13
SOURCE_ROW  = 32
SOURCE_COL  = 32
SOURCE_ADDR = SOURCE_ROW * 64 + SOURCE_COL

def probe_cell(row, col):
    """Read |E| magnitude at grid cell (row, col) from the completed buffer."""
    sel  = (stat.ch2_r() >> 5) & 1
    gpio = smag_a if sel == 0 else smag_b
    gpio.ch1_w((1 << 12) | (row * 64 + col))
    time.sleep(1e-5)
    return gpio.ch2_r() & 0xFFFF

def read_frame():
    """Read the full 64x64 magnitude frame from the completed ping-pong buffer."""
    sel  = (stat.ch2_r() >> 5) & 1
    gpio = smag_a if sel == 0 else smag_b
    frame = np.zeros(4096, dtype=np.uint16)
    for addr in range(4096):
        gpio.ch1_w((1 << 12) | addr)
        frame[addr] = gpio.ch2_r() & 0xFFFF
    return frame.reshape(64, 64)

print('✓ GPIO helpers ready')

---
## Cell 3 — Debug: Verify AXI GPIO bus is working

Reads back the ctrl GPIO after writing a known pattern.
If this fails, check the bitstream or AXI address map.

In [ ]:
# Write known patterns and read back (ctrl GPIO is ALL_OUTPUTS, so readback = what we wrote)
test_val_ch1 = (AMPLITUDE << 16) | PHASE_STEP
test_val_ch2 = (1 << 12) | SOURCE_ADDR

ctrl.ch1_w(test_val_ch1)
ctrl.ch2_w(test_val_ch2)
time.sleep(0.01)

rb_ch1 = ctrl.ch1_r()
rb_ch2 = ctrl.ch2_r()

print(f"CH1 wrote: 0x{test_val_ch1:08X}  read back: 0x{rb_ch1:08X}  {'OK' if rb_ch1 == test_val_ch1 else 'MISMATCH'}")
print(f"CH2 wrote: 0x{test_val_ch2:08X}  read back: 0x{rb_ch2:08X}  {'OK' if rb_ch2 == test_val_ch2 else 'MISMATCH'}")
print()

if rb_ch1 == test_val_ch1 and rb_ch2 == test_val_ch2:
    print('PASS — AXI GPIO ctrl is reachable, writes are correct')
    print(f'  phase_step = {rb_ch1 & 0xFFFF}')
    print(f'  amplitude  = {(rb_ch1 >> 16) & 0xFFFF}')
    print(f'  source_addr= {rb_ch2 & 0xFFF} (row={rb_ch2 & 0xFFF >> 6}, col={rb_ch2 & 0x3F})')
    print(f'  solver_en  = {(rb_ch2 >> 12) & 1}')
else:
    print('FAIL — GPIO readback mismatch. AXI bus may not be connected.')
    print('  Check: bitstream loaded? AXI addresses correct?')

---
## Cell 4 — Run solver continuously

**Important:** The FDTD solver runs ONE iteration (8192 cycles) then stops.
The PS must restart it after each iteration by toggling solver_enable low→high.
This cell runs 20 iterations and checks that the checksum changes each time.

In [ ]:
ITER_TIME = 0.004   # 8192 cycles at 25 MHz = 327 us; use 4ms to be safe

# CH2 bit map:
#   [11:0]  source_addr
#   [12]    solver_enable
#   [13]    mag_mode  (0=|E|, 1=Poynting)
#   [14]    sample_req  <- MUST be 1 to activate CORDIC; without it source_valid=0 and fields stay zero

ENABLE_BITS = (1 << 14) | (1 << 12)   # sample_req=1, solver_enable=1

def run_one_iteration():
    """Restart the solver for one FDTD iteration.
    Toggles solver_enable low->high to reset the counter.
    sample_req (bit 14) must be set or CORDIC never fires and source_valid stays 0."""
    ctrl.ch2_w(SOURCE_ADDR)                   # solver_enable=0, sample_req=0 -> counter resets
    time.sleep(1e-5)
    ctrl.ch2_w(ENABLE_BITS | SOURCE_ADDR)     # solver_enable=1, sample_req=1 -> CORDIC + solver active
    time.sleep(ITER_TIME)

# Prime the CORDIC: run a few iterations before checking checksums
# so held_source_valid latches to 1 and real field data starts flowing
print("Priming CORDIC (10 warm-up iterations)...")
for _ in range(10):
    run_one_iteration()

checksums = []
for i in range(20):
    run_one_iteration()
    checksums.append(stat.ch1_r())

unique = len(set(checksums))
print(f"Checksums over 20 iterations ({unique} unique):")
for i, c in enumerate(checksums):
    print(f"  iter {i+1:2d}: 0x{c:08X}")
print()
if unique >= 10:
    print(f'PASS — solver is running and updating fields ({unique}/20 unique checksums)')
elif unique >= 2:
    print(f'PARTIAL — solver running but checksums stabilising (steady-state oscillation is normal)')
else:
    print(f'FAIL — checksum not changing. Check GPIO debug cell above.')

---
## Cell 5 — Run N iterations then probe magnitude

In [ ]:
# Run 50 iterations to let the wave build up, then read the source cell
print("Running 50 FDTD iterations...")
for _ in range(50):
    run_one_iteration()

src  = probe_cell(SOURCE_ROW, SOURCE_COL)
near = probe_cell(SOURCE_ROW, SOURCE_COL + 8)
far  = probe_cell(SOURCE_ROW, SOURCE_COL + 20)
edge = probe_cell(1, 1)

print(f"source ({SOURCE_ROW},{SOURCE_COL})        = {src}")
print(f"near   ({SOURCE_ROW},{SOURCE_COL+8})       = {near}")
print(f"far    ({SOURCE_ROW},{SOURCE_COL+20})      = {far}")
print(f"edge   (1,1)              = {edge}")
print()

if src > 50:
    print('PASS — source cell has energy')
else:
    print('FAIL — source cell still zero. Check source_addr and amplitude.')
if near > 0:
    print('PASS — wave has propagated to nearby cell')
else:
    print('WARN — near cell still zero, may need more iterations')
if edge < max(src, 1) // 4:
    print('PASS — PML has absorbed wave at boundary')
else:
    print('WARN — boundary not well absorbed')

---
## Cell 6 — Read full 64×64 frame after steady state

In [ ]:
# Run to steady state (200 iterations)
print("Running 200 iterations to steady state...")
for _ in range(200):
    run_one_iteration()

print("Reading full 64x64 frame...")
frame = read_frame()

nonzero = np.count_nonzero(frame)
print(f"Max  |E| = {frame.max()}")
print(f"Mean |E| = {frame.mean():.1f}")
print(f"Nonzero  = {nonzero} / 4096 cells ({100*nonzero/4096:.1f}%)")
print(f"Source ({SOURCE_ROW},{SOURCE_COL}) = {frame[SOURCE_ROW, SOURCE_COL]}")
print()

if frame.max() > 0 and nonzero > 10:
    print('PASS — frame has field data')
else:
    print('FAIL — frame is empty')

---
## Cell 7 — Plot 64×64 magnitude field

In [ ]:
try:
    import matplotlib.pyplot as plt
    %matplotlib inline

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    im = axes[0].imshow(frame, cmap='hot', origin='lower', aspect='equal')
    axes[0].scatter([SOURCE_COL], [SOURCE_ROW], c='cyan', s=80, marker='x',
                    linewidths=2, label='source')
    axes[0].set_title('|E| magnitude — 64x64 FDTD (200 iterations)')
    axes[0].set_xlabel('column'); axes[0].set_ylabel('row')
    axes[0].legend()
    plt.colorbar(im, ax=axes[0], label='|E| (Q3.13)')

    axes[1].plot(frame[SOURCE_ROW, :], color='orangered', lw=2)
    axes[1].axvline(SOURCE_COL, color='cyan', linestyle='--', label='source')
    axes[1].set_title(f'|E| cross-section at row {SOURCE_ROW}')
    axes[1].set_xlabel('column'); axes[1].set_ylabel('|E| magnitude')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/home/xilinx/fdtd_result.png', dpi=150)
    plt.show()
    print('Plot saved to /home/xilinx/fdtd_result.png')

except ImportError:
    print('matplotlib not available')
    print('Row 32 cross-section:', frame[SOURCE_ROW, :])

---
## Cell 8 — Verify source location is configurable

In [ ]:
def run_at_source(row, col, iters=100):
    """Point source at (row,col), run iters iterations, probe that cell."""
    addr = row * 64 + col
    for _ in range(iters):
        ctrl.ch2_w(addr)                          # solver_enable=0
        time.sleep(1e-5)
        ctrl.ch2_w(ENABLE_BITS | addr)            # solver_enable=1, sample_req=1
        time.sleep(ITER_TIME)
    return probe_cell(row, col)

e_tl = run_at_source(16, 16)
e_br = run_at_source(48, 48)

# Restore centre
ctrl.ch2_w(ENABLE_BITS | SOURCE_ADDR)

print(f"Source at (16,16) -> |E| at (16,16) = {e_tl}")
print(f"Source at (48,48) -> |E| at (48,48) = {e_br}")
print()
if e_tl > 50 and e_br > 50:
    print('PASS — source location is runtime-configurable')
else:
    print('FAIL — one or both locations read zero')

---
## Cell 9 — Final status dump

In [ ]:
s2 = stat.ch2_r()
print("=== Status register ===")
print(f"  solver_checksum  = 0x{stat.ch1_r():08X}")
print(f"  solver_done      = {(s2>>0)&1}")
print(f"  source_valid     = {(s2>>1)&1}")
print(f"  mag_done         = {(s2>>2)&1}")
print(f"  mag_busy         = {(s2>>3)&1}")
print(f"  source_latched   = {(s2>>4)&1}")
print(f"  pp_read_sel      = {(s2>>5)&1}")
print(f"  pp_frame_ready   = {(s2>>6)&1}")
print(f"  source_q313      = {(s2>>7)&0xFFFF}")
print()
print("=== Register map ===")
print("  0x41200000  ctrl    CH1={amplitude,phase_step}  CH2={sample_req,mag_mode,solver_enable,source_addr}")
print("  0x41210000  status  CH1=solver_checksum  CH2={source_q313[22:7],pp_frame_ready[6],pp_read_sel[5],...}")
print("  0x41220000  smag_a  CH1={enb[12],addrb[11:0]}  CH2={doutb[15:0]}")
print("  0x41230000  smag_b  same")